# 🧩 Module D: GroupBy vs Pivot Table 深度对比

**核心问题**: 什么时候通过 `groupby`? 什么时候用 `pivot_table`?

*   **一维分析 (1D)**: `groupby` 是王者 (e.g., 每个部门的平均工资)。
*   **二维/多维分析 (2D+)**: `pivot_table` 是神器 (e.g., 不同部门在不同年份的工资变化)。

**本节目标**:
1.  体验 `groupby` + `unstack` 的痛苦。
2.  体验 `pivot_table` 的优雅。
3.  掌握 `margins` (总计) 和 `fill_value` (填充) 的用法。

---

In [1]:
import pandas as pd
import numpy as np

# 🛠️ 0. 生成销售数据 (无需修改)
np.random.seed(42)
data = {
    'Date': pd.date_range(start='2023-01-01', periods=20, freq='D'),
    'Region': np.random.choice(['North', 'South', 'East', 'West'], 20),
    'Category': np.random.choice(['Electronics', 'Furniture', 'Clothing'], 20),
    'Sales': np.random.randint(100, 1000, 20),
    'Profit': np.random.randint(10, 100, 20)
}
df = pd.DataFrame(data)
df.head()

,Date,Region,Category,Sales,Profit
0,2023-01-01,East,Furniture,786,18
1,2023-01-02,West,Electronics,662,99
2,2023-01-03,North,Furniture,975,62
3,2023-01-04,East,Furniture,666,11
4,2023-01-05,East,Furniture,343,93


### Q1: GroupBy 的局限 (The Long Format)
**任务**: 计算 **除了 'South' 以外**，每个 `Region` (行) 和 `Category` (列) 的 **平均 Sales**。
**要求**: 先用 `groupby` 做，看看结果长什么样。

In [2]:
# Your groupby solution here
df.columns = df.columns.str.lower()
summary = df.groupby(['region', 'category'])['sales'].mean().unstack()
summary = summary.reset_index('region')
summary[~(summary['region'] == 'South')]

category,region,Clothing,Electronics,Furniture
0,East,266.0,602.0,544.000000
1,North,487.0,417.0,975.000000
3,West,394.0,662.0,581.333333


### Q2: 艰难的变形 (Unstacking)
**任务**: 把 Q1 的结果变成一个 **二维表格** (Region 为索引，Category 为列)。
**技巧**: 对 Q1 的结果调用 `.unstack()`。

*观察: 是不是为了画个表，步骤有点多？*

In [3]:
# Your unstack solution here


### 🎓 教程: Pivot Table 怎么用？
这就好比 Excel 里的透视表，只需要填 4 个空：

```python
# 语法结构
df.pivot_table(
    index='...',      # 行 (你想在左边看到什么?)
    columns='...',    # 列 (你想在头上看到什么?)
    values='...',     # 值 (中间填什么数字?)
    aggfunc='mean'    # 算法 (默认是 mean, 也可以是 sum, count, max)
)
```

**优势**：不用 `unstack()`，不用 `reset_index()`，一步到位！

### Q3: Pivot Table 的优雅 (The Spreadsheet Way)
**任务**: 用 `pivot_table` 一步到位实现 Q2 的效果。
**要求**:
1.  `index='Region'`
2.  `columns='Category'`
3.  `values='Sales'`
4.  `aggfunc='mean'` (默认就是 mean，但写出来好习惯)

In [4]:
# Your pivot_table solution here
summary = df.pivot_table(
    index='region',
    columns='category',
    values='sales',
    aggfunc='mean'
)
summary

category,Clothing,Electronics,Furniture
region,,,
East,266.0,602.0,544.000000
North,487.0,417.0,975.000000
South,NaN,NaN,918.000000
West,394.0,662.0,581.333333


### Q4: 缺失值与总计 (Handling Gaps & Totals)
有时候某些 Region 可能没有卖某种 Category 的货，或者我们需要看总销售额。

**任务**: 再次做 Pivot Table，但这次：
1.  算 **总和 (sum)** 而不是平均。
2.  如果有空值 (`NaN`)，用 `0` 填充 (`fill_value=0`)。
3.  添加行/列的 **总计** (`margins=True`)。
4.  给总计列起个名字叫 'Total_Sales' (`margins_name='Total_Sales'`)。

In [5]:
# Your advanced pivot_table solution here
summary = df.pivot_table(
    index='region',
    columns='category',
    values='sales',
    aggfunc='sum',
    margins=True,
    margins_name='Total_sales'
).fillna(0)
summary

category,Clothing,Electronics,Furniture,Total_sales
region,,,,
East,266.0,1806.0,2720.0,4792
North,487.0,834.0,975.0,2296
South,0.0,0.0,918.0,918
West,788.0,662.0,1744.0,3194
Total_sales,1541.0,3302.0,6357.0,11200


### 🧠 思考题
如果你想同时看 `Sales` 的**总和** 和 `Profit` 的**平均值**，`pivot_table` 该怎么写？
*(提示: aggfunc 可以传一个字典)*

In [6]:
# Optional: Multi-aggregation pivot
summary = df.pivot_table(
    index='region',
    columns='category',
    values=['sales', 'profit'],
    aggfunc={'sales': 'sum', 'profit': 'mean'},
    margins=True,
    margins_name='Total_sales'
).fillna(0)
summary

profit                                       sales              \
category    Clothing Electronics  Furniture Total_sales Clothing Electronics   
region                                                                         
East            45.0   73.666667  30.400000   46.444444    266.0      1806.0   
North           13.0   66.500000  62.000000   52.000000    487.0       834.0   
South            0.0    0.000000  56.000000   56.000000      0.0         0.0   
West            37.0   99.000000  47.666667   52.666667    788.0       662.0   
Total_sales     33.0   75.500000  41.300000   49.900000   1541.0      3302.0   

                                   
category    Furniture Total_sales  
region                             
East           2720.0        4792  
North           975.0        2296  
South           918.0         918  
West           1744.0        3194  
Total_sales    6357.0       11200

In [8]:
# 1. 打平列名 (List Comprehension)
# 只有当你做了 Multi-aggregation Index 时才需要这步
if isinstance(summary.columns, pd.MultiIndex):
    summary.columns = [f'{col[0]}_{col[1]}' for col in summary.columns]

# 2. 重置索引
summary = summary.reset_index()

summary

,region,profit_Clothing,profit_Electronics,profit_Furniture,profit_Total_sales,sales_Clothing,sales_Electronics,sales_Furniture,sales_Total_sales
0,East,45.0,73.666667,30.400000,46.444444,266.0,1806.0,2720.0,4792
1,North,13.0,66.500000,62.000000,52.000000,487.0,834.0,975.0,2296
2,South,0.0,0.000000,56.000000,56.000000,0.0,0.0,918.0,918
3,West,37.0,99.000000,47.666667,52.666667,788.0,662.0,1744.0,3194
4,Total_sales,33.0,75.500000,41.300000,49.900000,1541.0,3302.0,6357.0,11200
